# AI Automation Doctor — Real Qwen3-1.7B benchmark on Kaggle T4

Runs the repository's unchanged 32-case evaluator against both the base model and the existing tool-calling PEFT adapter through a local OpenAI-compatible endpoint. No paid Hugging Face Job is required.

**Before Run All:** Kaggle → Settings → Accelerator → **GPU T4**.

In [ ]:
!git clone -q https://github.com/zubairz4far/ai-automation-doctor.git
%cd ai-automation-doctor
!pip install -q -e . peft accelerate fastapi uvicorn

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess, time, requests, pathlib

log_path = pathlib.Path('/kaggle/working/aad_qwen_server.log')
log_file = log_path.open('w')
server = subprocess.Popen(
    ['python', '-m', 'scripts.kaggle_model_server'],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

deadline = time.time() + 900
while time.time() < deadline:
    if server.poll() is not None:
        log_file.flush()
        raise RuntimeError(log_path.read_text()[-5000:])
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if r.ok:
            print(r.json())
            break
    except requests.RequestException:
        pass
    time.sleep(3)
else:
    raise TimeoutError('Model server did not become ready. Inspect ' + str(log_path))

In [ ]:
!python -m scripts.evaluate_ai_diagnosis \
  --api-base-url http://127.0.0.1:8000/v1 \
  --model Qwen/Qwen3-1.7B \
  --timeout-seconds 90 \
  --output evals/results/ai_diagnosis_qwen3_1.7b.json

!python -m scripts.evaluate_ai_diagnosis \
  --api-base-url http://127.0.0.1:8000/v1 \
  --model tool-calling \
  --timeout-seconds 90 \
  --output evals/results/ai_diagnosis_tool_calling_adapter.json

In [ ]:
import json, pathlib, pandas as pd

results = pathlib.Path('evals/results')
base = json.loads((results / 'ai_diagnosis_qwen3_1.7b.json').read_text())
adapter = json.loads((results / 'ai_diagnosis_tool_calling_adapter.json').read_text())

def row(label, d):
    return {
        'model': label,
        'cases': d['cases'],
        'ai_accuracy': d['ai_accuracy'],
        'schema_validity_rate': d['schema_validity_rate'],
        'provider_failure_rate': d['provider_failure_rate'],
        'accuracy_on_valid_outputs': d['ai_accuracy_on_valid_outputs'],
    }

table = pd.DataFrame([row('Qwen3-1.7B', base), row('tool-calling adapter', adapter)])
display(table)

comparison = {
    'dataset': base['dataset'],
    'cases': base['cases'],
    'base': row('Qwen3-1.7B', base),
    'adapter': row('tool-calling adapter', adapter),
    'adapter_minus_base_accuracy_pp': (adapter['ai_accuracy'] - base['ai_accuracy']) * 100,
    'adapter_minus_base_schema_validity_pp': (adapter['schema_validity_rate'] - base['schema_validity_rate']) * 100,
}
(results / 'ai_diagnosis_qwen3_vs_adapter_comparison.json').write_text(json.dumps(comparison, indent=2) + '\n')
print(json.dumps(comparison, indent=2))

## Outputs to bring back to GitHub

- `evals/results/ai_diagnosis_qwen3_1.7b.json`
- `evals/results/ai_diagnosis_tool_calling_adapter.json`
- `evals/results/ai_diagnosis_qwen3_vs_adapter_comparison.json`

After the first measured run, freeze release thresholds from the observed baseline instead of inventing them beforehand.